# Can persistent pressure pay for an aggressive round trip?

**MVP 1 · B1/B2/B4/B7; A2/A5; E1/E4–E7.** Establish an economic baseline on one
execution venue. Owner: researcher. Design refreshed 2026-09-23; awaiting empirical evidence.

The outcome is a complete, costed trading decision with a substantial no-trade region.
Use the [opportunity map](opportunity-map.ipynb) and freeze the [protocol](../docs/research/study-protocol.md).

**Planned, not started:** research begins only after the infrastructure passes QR onboarding
acceptance. See the [QR delivery milestones](../docs/research/program.md#post-infrastructure-research-milestones).

## 1. Hypothesis and alternatives

**Mechanism:** persistent bid/ask pressure contains information about a subsequent move
that remains available after feasible delay. **Alternatives:** the feature just describes
recent returns; temporary displayed depth disappears; the effect is bid–ask mechanics or
bad data; gross prediction is too small to pay for entry and exit.

Start with depth imbalance, observed OFI, spread, recent returns and volatility. Add deeper
shape as an ablation. Define quantity units, depth bands, trailing windows and zero-depth
handling. Persistent means observed before the decision. Predict subsequent outcomes,
not the contemporaneous price change used to construct OFI.

## 2. Freeze the policy and comparison

| Component | Specify before fitting |
| --- | --- |
| Information | Messages received before the decision, staleness limits, feature windows |
| Entry | Forecast must exceed an explicit cost/risk margin; otherwise no trade |
| Sizing | Quantity and depth-participation cap; capital and inventory limits |
| Timing | Observed delay components and stress cases; arrival-time execution |
| Exit | Fixed horizon initially; timeout, partial execution and failed-exit rules |
| Overlap | How a new signal affects an existing position or outstanding order |
| Eligibility | Historical product access and permitted sides; no synthetic shorting of unsupported spot |

Compare no-trade, recent-return and simple-imbalance rules first. Fit a small linear/logistic
model if useful, then a tree challenger. Keep sample, costs, capital, risk and search budgets
comparable. Add OFI, persistence and deeper shape separately. Retain every attempted variant.
A sequence model needs an identified failure of the simpler alternatives.

## 3. Separate labels from the economic evaluation

Use future executable return or another target justified by the action. Directional accuracy
and midpoint markout are secondary diagnostics. A high accuracy classifier may have no
valuable trades. Check calibration specifically in the selected trading region.

Fit only on earlier data; purge overlapping information intervals at boundaries. Select
thresholds with validation, freeze, then evaluate untouched test once. Use the same
versioned policy and fitted state at each historical decision. A full-history fit cannot be
applied retroactively.

Cost complete round trips at entry and exit arrival times. Include financing where applicable,
forced liquidation and residual positions. Compare net P&L per eligible opportunity as well
as per executed trade, frequency, tails, daily exposure, size and delay sensitivity.

## 4. Supply matched daily policy outcomes

This worksheet consumes daily summaries from a retained policy evaluation; it does not
fit models or simulate orders. Choose **one instrument, quote currency, size, delay,
capital/risk configuration and evaluation partition per table**. Retain those in `study_ref`.
Use multiple runs of the worksheet for sensitivity scenarios and preserve every attempt.

Include every eligible day for every selected policy, including zero-trade days. `opportunities`
is the common predeclared decision set; `trades` is completed round trips, not level fills.
`net_pnl_quote` includes all costs and terminal liquidation under the declared rule.
Per-trade tail loss and the full inventory path still need inspection in the underlying report.
This first policy permits at most one completed round trip per eligible opportunity.

In [ ]:
# standard library imports
from math import isfinite

# third party library imports
import pandas as pd

study_ref = ""  # Retained study identity including instrument, currency, scenario and split.
policies = ["no_trade", "recent_return", "imbalance"]  # Extend only as predeclared.
daily_results: list[dict[str, object]] = []
columns = [
    "day",
    "policy",
    "opportunity_set_id",
    "opportunities",
    "trades",
    "net_pnl_quote",
    "capital_quote",
    "max_abs_inventory_quote",
]
daily = pd.DataFrame(daily_results, columns=columns)

In [ ]:
if daily.empty:
    print("AWAITING EVIDENCE — no policy result or model advantage is established.")
else:
    if (
        not study_ref.strip()
        or daily.isna().any().any()
        or daily.duplicated(["day", "policy"]).any()
    ):
        raise ValueError("Supply a retained study and complete unique day/policy rows.")
    expected = set(policies)
    for _, day_rows in daily.groupby("day"):
        if set(day_rows.policy) != expected or day_rows.opportunity_set_id.nunique() != 1:
            raise ValueError("Compare the same policy set and opportunities on every day.")
        if day_rows.opportunities.nunique() != 1 or day_rows.capital_quote.nunique() != 1:
            raise ValueError("Opportunity counts and allocated capital must match within each day.")
    for field in columns[3:]:
        daily[field] = pd.to_numeric(daily[field], errors="raise")
    if (daily[["opportunities", "trades", "max_abs_inventory_quote"]] < 0).any().any() or (
        daily.capital_quote <= 0
    ).any():
        raise ValueError("Counts/inventory must be nonnegative and capital positive.")
    if not daily[columns[3:]].map(isfinite).all().all():
        raise ValueError("Daily measurements must be finite.")
    if (daily[["opportunities", "trades"]] % 1 != 0).any().any() or (
        daily.trades > daily.opportunities
    ).any():
        raise ValueError("Counts must be integers, with at most one round trip per opportunity.")
    no_trade = daily[daily.policy == "no_trade"]
    if (no_trade[["trades", "net_pnl_quote", "max_abs_inventory_quote"]] != 0).any().any():
        raise ValueError("The flat, unfinanced no-trade baseline must remain flat with zero P&L.")
    summary = daily.groupby("policy").agg(
        days=("day", "nunique"),
        opportunities=("opportunities", "sum"),
        trades=("trades", "sum"),
        net_pnl_quote=("net_pnl_quote", "sum"),
        worst_day_quote=("net_pnl_quote", "min"),
        peak_inventory_quote=("max_abs_inventory_quote", "max"),
    )
    summary["net_per_opportunity_quote"] = summary.net_pnl_quote / summary.opportunities.where(
        summary.opportunities > 0
    )
    summary["net_per_trade_quote"] = summary.net_pnl_quote / summary.trades.where(
        summary.trades > 0
    )
    paired = daily.pivot(index="day", columns="policy", values="net_pnl_quote")
    paired_vs_recent_return = paired.subtract(paired["recent_return"], axis=0)
    display(summary)
    display(paired_vs_recent_return.rename_axis(columns="daily net improvement over recent_return"))

## 5. Challenge the apparent winner

Review daily paired improvements, not just aggregate P&L. Estimate uncertainty with blocks
appropriate to dependence and overlapping positions; inspect volatility/liquidity regimes,
directional exposure, extreme days and all trials in the family. The table above supplies
no confidence interval and does not correct for model selection.

Repeat at feasible and stressed delays, larger sizes and alternative justified cost regimes.
Check that execution failures and abstentions remain in denominators. Compare adding pressure
with recent returns held constant. Test whether pressure is more useful as a veto or quote
adjustment if it cannot fund an aggressive round trip.

**Reject/narrow:** no incremental OOS net value; edge gone at feasible delay; negligible
capacity; tiny, repeatedly optimized subsets; or profit concentrated in invalid episodes.

## 6. Decision and next experiment

Complete the [memo](../docs/research/study-protocol.md#decision-memo), including net per opportunity/trade,
frequency, worst days, inventory, size/delay sensitivity, uncertainty and trial count.
**Current conclusion: insufficient evidence.**

Use [liquidity shocks](liquidity-shocks.ipynb) as the focused alternative on the same sample.
Only after a credible local baseline should D2 ask whether received information from a
second venue improves the same decisions. Keep two-venue execution as a separate question.